# 02 - Baselines

Two references, neither of which counts as one of the three graded models:

1. **Most frequent class** - always predicts "real". Proves the accuracy trap.
2. **TF-IDF -> Logistic Regression** - a genuine, and genuinely strong, bar to clear.

**Protocol used in every model notebook:** train on train, tune the decision
threshold on **validation**, and report on **test** at that frozen threshold. Test
is never used to make a choice.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether this runs from notebooks/ or the project root.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config, evaluation, features, models, preprocessing

config.set_seed()
config.ensure_dirs()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
train, val, test = preprocessing.load_splits()
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:>5}: {len(part):>6,} rows, {part[config.TARGET].mean():.2%} fraudulent")

y_train, y_val, y_test = (part[config.TARGET].to_numpy() for part in (train, val, test))

## 1. Most-frequent baseline

In [ ]:
dummy = models.train_most_frequent_baseline(train[[config.FULL_TEXT_COLUMN]], y_train)

# Train and validation are evaluated separately, never in one wrapped call.
dummy_train_metrics = evaluation.evaluate_predictions(
    y_train, models.predict_proba(dummy, train[[config.FULL_TEXT_COLUMN]]))
dummy_val_metrics = evaluation.evaluate_predictions(
    y_val, models.predict_proba(dummy, val[[config.FULL_TEXT_COLUMN]]))

print("train:", {k: round(v, 4) for k, v in dummy_train_metrics.items()})
print("val:  ", {k: round(v, 4) for k, v in dummy_val_metrics.items()})
print(f"\n-> {dummy_val_metrics['accuracy']:.2%} accuracy, recall {dummy_val_metrics['recall']:.0%}. "
      "Every scam missed.")

evaluation.log_experiment("baseline_most_frequent", dummy_val_metrics, split="val",
                          notes="always predicts the majority class")

## 2. TF-IDF -> Logistic Regression

`class_weight="balanced"` so the minority class is not simply ignored.

In [ ]:
logreg = models.train_tfidf_logreg(train[config.FULL_TEXT_COLUMN], y_train)

proba_train = models.predict_proba(logreg, train[config.FULL_TEXT_COLUMN])
proba_val = models.predict_proba(logreg, val[config.FULL_TEXT_COLUMN])

logreg_train_metrics = evaluation.evaluate_predictions(y_train, proba_train)
logreg_val_metrics = evaluation.evaluate_predictions(y_val, proba_val)

print("train:", {k: round(v, 4) for k, v in logreg_train_metrics.items()})
print("val:  ", {k: round(v, 4) for k, v in logreg_val_metrics.items()})
print(f"\ngap in AP (train - val): "
      f"{logreg_train_metrics['average_precision'] - logreg_val_metrics['average_precision']:.3f}"
      "  <- large gap means it is memorising")

### Tune the threshold on validation

In [ ]:
best_threshold, best_f1 = evaluation.tune_threshold(y_val, proba_val, beta=1.0)
print(f"best F1 threshold on val: {best_threshold:.3f}  (F1={best_f1:.3f})")

recall_threshold, recall_at_precision = evaluation.threshold_for_precision(
    y_val, proba_val, min_precision=0.9)
print(f"to hold 90% precision: threshold {recall_threshold:.3f}, "
      f"catching {recall_at_precision:.1%} of scams")

logreg_val_tuned = evaluation.evaluate_predictions(y_val, proba_val, threshold=best_threshold)
print("\nval @0.5:  ", {k: round(v, 3) for k, v in logreg_val_metrics.items()})
print("val @tuned:", {k: round(v, 3) for k, v in logreg_val_tuned.items()})

evaluation.log_experiment("baseline_tfidf_logreg", logreg_val_tuned, split="val",
                          notes=f"threshold tuned on val (F1), C={config.LOGREG_PARAMS['C']}")

### Report on test at the frozen threshold

In [ ]:
proba_test = models.predict_proba(logreg, test[config.FULL_TEXT_COLUMN])
logreg_test_metrics = evaluation.evaluate_predictions(y_test, proba_test, threshold=best_threshold)
print("test:", {k: round(v, 4) for k, v in logreg_test_metrics.items()})

evaluation.log_experiment("baseline_tfidf_logreg", logreg_test_metrics, split="test",
                          notes="threshold frozen from val")
models.save_test_predictions("baseline_tfidf_logreg", y_test, proba_test)

In [ ]:
ax = evaluation.plot_pr_curve(y_test, proba_test, label="TF-IDF + LogReg")
evaluation.plot_pr_curve(y_test, models.predict_proba(dummy, test[[config.FULL_TEXT_COLUMN]]),
                         label="most frequent", ax=ax, save_as="07_baseline_pr_curve.png")
plt.show()

evaluation.plot_confusion_matrix(y_test, (proba_test >= best_threshold).astype(int),
                                 title=f"TF-IDF + LogReg (test, t={best_threshold:.2f})",
                                 save_as="08_baseline_confusion_matrix.png")
plt.show()

### Which words drive the decision?

In [ ]:
vectorizer = logreg.named_steps["tfidf"]
coefficients = logreg.named_steps["logreg"].coef_[0]
vocabulary = vectorizer.get_feature_names_out()

order = np.argsort(coefficients)
top_fake = pd.Series(coefficients[order[-20:]], index=vocabulary[order[-20:]])
top_real = pd.Series(coefficients[order[:20]], index=vocabulary[order[:20]])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
top_fake.plot.barh(ax=axes[0], color="#c44e52")
axes[0].set_title("Pushes towards FAKE")
top_real.sort_values(ascending=False).plot.barh(ax=axes[1], color="#4c72b0")
axes[1].set_title("Pushes towards REAL")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "09_baseline_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

## Bar to clear

| model | what it shows |
|---|---|
| most frequent | ~95% accuracy, 0% recall. Accuracy is a lie. |
| TF-IDF + LogReg | The number the three graded models must beat. |

Run `evaluation.load_experiments()` at any point to see the log so far.

Next: `03_models_lightgbm.ipynb`.

In [ ]:
evaluation.load_experiments()